# IAM-811 — Servidor Unificado de Inferencia en RunPod y Google Colab (M1 + M2 + M3 + M4)
**Estudiante:** Amilkar Hugo Guarachi Nina · **Docente:** Ing. Alvaro Pinto Mamani

Siguiendo la estructura oficial de los notebooks `week_2_servidor_pose_runpod.ipynb` y `week_2_servidor_pose_colab_v2.ipynb`, este cuaderno levanta el **Endpoint Único de Inferencia** que integra:
- **M1:** Detección de placas (`YOLOv9-t ONNX`) + OCR (`FastPlateOCR CCT ONNX` + `EasyOCR`).
- **M2:** Segmentación de instancias de vehículos (`YOLO11s-seg`: `moto`, `auto`, `minibus`, `camion`, `bus`).
- **M3:** Keypoints de llantas (`YOLO11s-pose`, `m3_ruedas_pose.pt`) → Estimación del número de ejes.
- **M4 (Bono +5 pts):** Segmentación y conteo anónimo de pasajeros dentro del vehículo.

Funciona **igual en RunPod** (usando el proxy nativo `https://{RUNPOD_POD_ID}-8000.proxy.runpod.net`) **y en Google Colab** (creando un túnel HTTPS gratuito con `trycloudflare.com`).

In [ ]:
# 1. Instalación de dependencias y túnel Cloudflare
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| CUDA:', torch.version.cuda)

!pip -q install fastapi "uvicorn[standard]" python-multipart opencv-python-headless onnxruntime-gpu ultralytics fast-plate-ocr easyocr
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. Subir paquete_inferencia_cloud.zip (incluye main.py, diagnostico_entorno.py, M1 ONNX, FastPlateOCR y M3 m3_ruedas_pose.pt)
# También puedes subir m2_vehiculos_seg.pt si lo entrenaste aparte.
import os, shutil
from google.colab import files

os.makedirs('/content/inferencia/modelos', exist_ok=True)
print('Sube paquete_inferencia_cloud.zip (y opcionalmente m2_vehiculos_seg.pt):')
for nombre in files.upload().keys():
    if nombre.lower().endswith('.zip'):
        !unzip -qo "$nombre" -d /content/inferencia
    elif nombre.lower().endswith('.pt') or nombre.lower().endswith('.onnx'):
        shutil.move(nombre, '/content/inferencia/modelos/' + nombre)

%cd /content/inferencia
!python diagnostico_entorno.py --servicio cloud-gpu-inferencia

In [ ]:
# 3. Iniciar el Servidor FastAPI en segundo plano y exponer URL_PUBLICA (RunPod o Cloudflare Tunnel)
import os, re, time, threading, subprocess, uvicorn
from main import app

PUERTO = 8000
if 'servidor' in globals() and servidor.started and not servidor.should_exit:
    print('El servidor ya está en ejecución.')
else:
    config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
    servidor = uvicorn.Server(config)
    hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
    hilo_servidor.start()
    for _ in range(50):
        if servidor.started:
            break
        time.sleep(0.1)
    print('Servidor FastAPI iniciado:', servidor.started)

POD_ID = os.environ.get('RUNPOD_POD_ID')
if POD_ID:
    URL_PUBLICA = f'https://{POD_ID}-{PUERTO}.proxy.runpod.net'
else:
    URL_PUBLICA = None
    proceso_tunel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PUERTO}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    log_tunel = []
    def _leer_tunel():
        for linea in proceso_tunel.stdout:
            log_tunel.append(linea)
    threading.Thread(target=_leer_tunel, daemon=True).start()
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log_tunel))
        if m:
            URL_PUBLICA = m.group(0)
            break
        time.sleep(0.5)
    time.sleep(3)

print('====================================================================')
print('COPIA ESTA URL EN LA VARIABLE INFERENCE_URL DE TU APLICACIÓN WEB:')
print('INFERENCE_URL=' + str(URL_PUBLICA))
print('====================================================================')

In [ ]:
# 4. Prueba en vivo del Endpoint Único (/health y /infer) sobre imágenes reales con placa y visualización
import os, json, urllib.request, base64, cv2, numpy as np
import matplotlib.pyplot as plt

salud = json.loads(urllib.request.urlopen(f'http://localhost:{PUERTO}/health', timeout=15).read())
print('Estado /health:', json.dumps(salud, indent=2, ensure_ascii=False))

pruebas = sorted([f'/content/inferencia/imagenes_prueba/{f}' for f in os.listdir('/content/inferencia/imagenes_prueba') if f.endswith('.jpg')])[:4]
if not pruebas:
    !wget -q -O /content/prueba_bus.jpg https://ultralytics.com/images/bus.jpg
    pruebas = ['/content/prueba_bus.jpg']

fig, axs = plt.subplots(1, len(pruebas), figsize=(6 * len(pruebas), 5), squeeze=False)
for idx, ruta_p in enumerate(pruebas):
    b64_img = base64.b64encode(open(ruta_p, 'rb').read()).decode()
    payload = json.dumps({'imagen_base64': b64_img, 'umbral_confianza': 0.25}).encode()
    req = urllib.request.Request(
        f'http://localhost:{PUERTO}/infer/json',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    res = json.loads(urllib.request.urlopen(req, timeout=60).read())
    print(f'=== Resultado en {os.path.basename(ruta_p)} ===')
    for k in ('placa', 'confianza', 'categoria', 'ejes', 'pasajeros', 'latencia', 'placa_detalle', 'vehiculo', 'keypoints', 'desglose_latencia_ms'):
        print(f'  {k}: {res.get(k)}')
    if res.get('imagen_evidencia_b64'):
        raw_vis = base64.b64decode(res['imagen_evidencia_b64'].split(',')[-1])
        im_vis = cv2.imdecode(np.frombuffer(raw_vis, np.uint8), cv2.IMREAD_COLOR)
        axs[0][idx].imshow(cv2.cvtColor(im_vis, cv2.COLOR_BGR2RGB))
        axs[0][idx].set_title(f"{res.get('placa') or 'N/A'} | {res.get('categoria')} | {res.get('ejes')} ejes ({res.get('latencia')} ms)", fontsize=10)
        axs[0][idx].axis('off')
plt.tight_layout()
plt.show()